# Этап 1: АУДИТ справочника курортов

Аудит справочника курортов: проверка  32 записей , уникальность resort_id, регионы, координаты.

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Корень проекта — БЕЗ хардкода абсолютных путей внутри кода,
# путь задаётся один раз здесь.
#BASE = Path(r"C:\Users\dak\OneDrive\Документы\3 Горнолыжные курорты\Travel_Designer_Resort_Ranking")
BASE = Path (".")

RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## 1.Справочник 32 курортов

In [2]:
with open(RAW / "resorts.json", encoding="utf-8-sig") as f:
    resorts_raw = json.load(f)

# Проверка структуры
print("type:", type(resorts_raw))
print("элементов:", len(resorts_raw))

# Структура — список списков: raw[0] = заголовки, raw[1:] = данные
resorts_df = pd.DataFrame(resorts_raw[1:], columns=resorts_raw[0])

print("\nФорма:", resorts_df.shape)
print("Колонки:", resorts_df.columns.tolist())
resorts_df.head()

type: <class 'list'>
,элементов: 33
,
,Форма: (32, 33)
,Колонки: ['resort_id', 'name_ru', 'region', 'latitude', 'longitude', 'coordinate_quality', 'coordinate_source_title', 'coordinate_source_url', 'wikidata_id', 'piste_segments_6km', 'lift_lines_6km', 'lodging_objects_6km', 'food_objects_6km', 'transport_objects_6km', 'attraction_objects_6km', 'osm_radius_km', 'osm_overlap_warning', 'open_meteo_elevation_m', 'winter_mean_temperature_c', 'avg_snowfall_cm_per_season', 'avg_freezing_days_per_season', 'avg_snow_days_per_season', 'max_daily_snowfall_cm', 'weather_model', 'weather_period_start', 'weather_period_end', 'nearest_scheduled_airport', 'nearest_airport_iata', 'nearest_airport_distance_km', 'nearest_airport_municipality', 'completeness_pct', 'quality_flags', 'collected_at']


,resort_id,name_ru,region,latitude,longitude,coordinate_quality,coordinate_source_title,coordinate_source_url,wikidata_id,piste_segments_6km,lift_lines_6km,lodging_objects_6km,food_objects_6km,transport_objects_6km,attraction_objects_6km,osm_radius_km,osm_overlap_warning,open_meteo_elevation_m,winter_mean_temperature_c,avg_snowfall_cm_per_season,avg_freezing_days_per_season,avg_snow_days_per_season,max_daily_snowfall_cm,weather_model,weather_period_start,weather_period_end,nearest_scheduled_airport,nearest_airport_iata,nearest_airport_distance_km,nearest_airport_municipality,completeness_pct,quality_flags,collected_at
0,R001,Роза Хутор,Краснодарский край,43.646390,40.332220,nearby_feature_proxy,Роза Хутор,https://ru.wikipedia.org/wiki/%D0%A0%D0%BE%D0%...,Q1335490,334,64,86,107,45,42,6,"Possible overlap with Газпром Поляна (6.0 km),...",950,4.272848,390.082,35.2,92.6,47.67,ERA5,2021-11-01,2026-04-30,Sochi International Airport,AER,37.3,Sochi,1.0,coordinates are proxy; OSM radii overlap,46279.509709
1,R002,Красная Поляна,Краснодарский край,43.686110,40.260000,resort_or_complex_page,Красная Поляна (курорт),https://ru.wikipedia.org/wiki/%D0%9A%D1%80%D0%...,Q120478823,121,44,255,259,76,82,6,"Possible overlap with Газпром Поляна (1.6 km),...",513,7.114349,390.082,16.8,92.6,47.67,ERA5,2021-11-01,2026-04-30,Sochi International Airport,AER,35.9,Sochi,1.0,OSM radii overlap,46279.509709
2,R003,Газпром Поляна,Краснодарский край,43.684519,40.280172,resort_or_complex_page,Курорт Газпрома,https://ru.wikipedia.org/wiki/%D0%9A%D1%83%D1%...,Q4144556,240,63,223,243,77,77,6,"Possible overlap with Красная Поляна (1.6 km),...",546,6.900331,390.082,18.6,92.6,47.67,ERA5,2021-11-01,2026-04-30,Sochi International Airport,AER,36.9,Sochi,1.0,OSM radii overlap,46279.509709
3,R004,Шерегеш,Кемеровская область,52.953380,87.955480,resort_or_complex_page,Шерегеш (горнолыжный курорт),https://ru.wikipedia.org/wiki/%D0%A8%D0%B5%D1%...,Q21645716,142,23,233,57,54,4,6,None,695,-9.152539,243.922,151.6,93.2,21.07,ERA5,2021-11-01,2026-04-30,Spichenkovo Airport,NOZ,119.2,Novokuznetsk,1.0,None,46279.509709
4,R005,Архыз,Карачаево-Черкесская Республика,43.543330,41.181670,resort_or_complex_page,Архыз (курорт),https://ru.wikipedia.org/wiki/%D0%90%D1%80%D1%...,Q4127579,42,13,187,69,18,25,6,None,1655,0.470088,523.194,81.6,107.6,29.47,ERA5,2021-11-01,2026-04-30,Sochi International Airport,AER,99.4,Sochi,1.0,None,46279.509709


In [3]:
#  Перечень курортов
unique_resorts = resorts_df["name_ru"].unique()
display(unique_resorts)

array(['Роза Хутор', 'Красная Поляна', 'Газпром Поляна', 'Шерегеш',
       'Архыз', 'Домбай', 'Эльбрус', 'Чегет', 'Мамисон', 'Цей',
       'Манжерок', 'Телецкий', 'Большой Вудъявр', 'Кукисвумчорр',
       'КолаСпортЛэнд', 'Горный воздух', 'Холдоми', 'Морозная гора',
       'Бобровый лог', 'Соболиная гора', 'Гора Белая', 'Евразия',
       'Аджигардак', 'Солнечная долина', 'Завьялиха', 'Абзаково',
       'Мраткино', 'Губаха', 'Игора', 'Сорочаны', 'Красная Глинка',
       'Гладенькая'], dtype=object)

In [4]:
# --- Пометки курортов, требующих особого внимания ---
# Формат: {resort_id: {"related_resort_id": ..., "note": "..."}}
resort_notes = {
    "R015": {
        "related_resort_id": "R013", # Большой Вудъявр
        "note": "Историческое название части курорта «Большой Вудъявр». "
                "Отдельного юридического лица не существует. "
                "Точка может дублировать зону катания R013.",
    },
}

# --- source_name: сохраняем исходное написание ---
# В нашем случае ни одно название не менялось, поэтому source_name = name_ru
resorts_df["source_name"] = resorts_df["name_ru"]

# --- related_resort_id ---
resorts_df["related_resort_id"] = resorts_df["resort_id"].map(
    lambda x: resort_notes.get(x, {}).get("related_resort_id")
)

# --- audit_note ---
resorts_df["audit_note"] = resorts_df["resort_id"].map(
    lambda x: resort_notes.get(x, {}).get("note")
)

# --- флаг в quality_flags ---
def merge_flag(flags, extra):
    if pd.isna(flags) or flags == "":
        return extra
    if extra in flags:
        return flags
    return f"{flags}; {extra}"

resorts_df["quality_flags"] = resorts_df.apply(
    lambda row: merge_flag(row["quality_flags"], "historical name; part of another resort")
                if row["resort_id"] in resort_notes else row["quality_flags"],
    axis=1,
)

# --- проверка ---
print("R015 после пометки:")
print(resorts_df.loc[resorts_df["resort_id"] == "R015",
                     ["resort_id", "name_ru", "region",
                      "related_resort_id", "audit_note", "quality_flags"]].T)

R015 после пометки:
,                                                                  14
,resort_id                                                       R015
,name_ru                                                КолаСпортЛэнд
,region                                            Мурманская область
,related_resort_id                                               R013
,audit_note         Историческое название части курорта «Большой В...
,quality_flags      check coordinates; OSM radii overlap; historic...


In [5]:
# 1. Все audit_note-записи ссылаются на существующий resort_id
related = resorts_df["related_resort_id"].dropna().unique()
existing = set(resorts_df["resort_id"])
missing = [r for r in related if r not in existing]
assert not missing, f"related_resort_id ссылается на несуществующие: {missing}"
print(f"✅ related_resort_id корректен ({len(related)} связей)")

# 2. Ни у одного курорта нет двух conflicting related_resort_id
assert len(resorts_df) == 32, "Изменилось число строк!"
print(f"✅ Всё ещё 32 курорта")

# 3. Сводка помеченных курортов
flagged = resorts_df[resorts_df["audit_note"].notna()]
print(f"\nКурортов с audit_note: {len(flagged)}")
print(flagged[["resort_id", "name_ru", "related_resort_id"]].to_string(index=False))


✅ related_resort_id корректен (1 связей)
,✅ Всё ещё 32 курорта
,
,Курортов с audit_note: 1
,resort_id       name_ru related_resort_id
,     R015 КолаСпортЛэнд              R013


## Аудит справочника курортов

In [6]:
# 1. Уникальность resort_id
assert resorts_df["resort_id"].is_unique, "resort_id не уникален!"
assert len(resorts_df) == 32, f"Ожидалось 32 курорта, получили {len(resorts_df)}"
print("✅ 32 курорта, resort_id уникальны")

# 2. Число регионов
n_regions = resorts_df["region"].nunique()
print(f"Уникальных регионов: {n_regions}")

# 3. Диапазоны координат
lat = pd.to_numeric(resorts_df["latitude"], errors="coerce")
lon = pd.to_numeric(resorts_df["longitude"], errors="coerce")
assert lat.between(-90, 90).all(), "Широта вне диапазона"
assert lon.between(-180, 180).all(), "Долгота вне диапазона"
print("✅ Координаты в допустимых диапазонах")
# 4. Качество координат
print("\nКачество координат:")
print(resorts_df["coordinate_quality"].value_counts())

# 5. Пропуски по ключевым полям
key_cols = ["resort_id", "name_ru", "region", "latitude", "longitude", "coordinate_quality"]
print("\nПропуски по ключевым полям:")
print(resorts_df[key_cols].isna().sum())

# 6. Сводка по регионам
print("\nКурортов по регионам:")
print(resorts_df.groupby("region")["resort_id"].count().sort_values(ascending=False))

✅ 32 курорта, resort_id уникальны
,Уникальных регионов: 20
,✅ Координаты в допустимых диапазонах
,
,Качество координат:
,manual_review_required    18
,nearby_feature_proxy       8
,resort_or_complex_page     6
,Name: coordinate_quality, dtype: int64
,
,Пропуски по ключевым полям:
,resort_id             0
,name_ru               0
,region                0
,latitude              0
,longitude             0
,coordinate_quality    0
,dtype: int64
,
,Курортов по регионам:
,region
,Челябинская область                    4
,Краснодарский край                     3
,Мурманская область                     3
,Республика Северная Осетия — Алания    2
,Карачаево-Черкесская Республика        2
,Кабардино-Балкарская Республика        2
,Республика Алтай                       2
,Республика Башкортостан                2
,Хабаровский край                       1
,Свердловская область                   1
,Сахалинская область                    1
,Самарская область                      1
,Республика Хакаси

## 2. Месячные цены 

In [7]:
with open(RAW / "prices_monthly.json", encoding="utf-8-sig") as f:
    prices_raw = json.load(f)

print("type:", type(prices_raw))
print("элементов:", len(prices_raw))

# Тот же формат: список списков
prices_month_df = pd.DataFrame(prices_raw[1:], columns=prices_raw[0])

print("\nФорма:", prices_month_df.shape)
print("Колонки:", prices_month_df.columns.tolist())
prices_month_df.head(10)

type: <class 'list'>
,элементов: 37
,
,Форма: (36, 12)
,Колонки: ['resort_id', 'resort_name', 'month', 'adult_price_min_rub', 'adult_price_max_rub', 'child_price_min_rub', 'child_price_max_rub', 'tariff_product', 'coverage', 'source_quality', 'source_url', 'checked_at']


,resort_id,resort_name,month,adult_price_min_rub,adult_price_max_rub,child_price_min_rub,child_price_max_rub,tariff_product,coverage,source_quality,source_url,checked_at
0,R001,Роза Хутор,2026-01,5550,6350,2800.0,3200.0,Ски-пасс на 1 день,05–31.01.2026; тариф менялся внутри месяца,secondary,https://skipasskp.ru/skipass-ceny-kp,46279
1,R001,Роза Хутор,2026-02,6800,6800,3400.0,3400.0,Ски-пасс на 1 день,01–28.02.2026,secondary,https://skipasskp.ru/skipass-ceny-kp,46279
2,R001,Роза Хутор,2026-03,5550,6800,2800.0,3400.0,Ски-пасс на 1 день,01–31.03.2026; изменение тарифа 10.03,secondary,https://skipasskp.ru/skipass-ceny-kp,46279
3,R001,Роза Хутор,2026-04,5190,5190,2600.0,2600.0,Ски-пасс на 1 день,01–12.04.2026,secondary,https://skipasskp.ru/skipass-ceny-kp,46279
4,R002,Красная Поляна,2025-12,3900,5250,2300.0,3050.0,Ски-пасс на 1 день,26–31.12.2025,secondary,https://skipasskp.ru/skipass-ceny-kp,46279
5,R002,Красная Поляна,2026-01,4100,5250,2500.0,3050.0,Ски-пасс на 1 день,01–31.01.2026,secondary,https://skipasskp.ru/skipass-ceny-kp,46279
6,R002,Красная Поляна,2026-02,5200,5200,2750.0,2750.0,Ски-пасс на 1 день,01–28.02.2026,secondary,https://skipasskp.ru/skipass-ceny-kp,46279
7,R002,Красная Поляна,2026-03,4100,5200,2500.0,2750.0,Ски-пасс на 1 день,01–31.03.2026; изменение тарифа 10.03,secondary,https://skipasskp.ru/skipass-ceny-kp,46279
8,R002,Красная Поляна,2026-04,3900,3900,2300.0,2300.0,Ски-пасс на 1 день,01–30.04.2026,secondary,https://skipasskp.ru/skipass-ceny-kp,46279
9,R002,Красная Поляна,2026-05,3900,3900,2300.0,2300.0,Ски-пасс на 1 день,01–11.05.2026,secondary,https://skipasskp.ru/skipass-ceny-kp,46279


## Аудит цен

In [8]:
 #Приведение к числу
for col in ["adult_price_min_rub", "adult_price_max_rub",
            "child_price_min_rub", "child_price_max_rub"]:
    prices_month_df[col] = pd.to_numeric(prices_month_df[col], errors="coerce")

# 1. Цены положительны
assert (prices_month_df["adult_price_min_rub"].dropna() > 0).all()
assert (prices_month_df["adult_price_max_rub"].dropna() > 0).all()

# 2. min <= max
mask = prices_month_df["adult_price_min_rub"].notna() & prices_month_df["adult_price_max_rub"].notna()
assert (prices_month_df.loc[mask, "adult_price_min_rub"] <= prices_month_df.loc[mask, "adult_price_max_rub"]).all()
print("✅ Цены положительны, min <= max")

# 3. Дубликаты
dups = prices_month_df.duplicated(subset=["resort_id", "month", "tariff_product"]).sum()
print(f"Дубликатов по (resort_id, month, tariff_product): {dups}")

# 4. Сколько курортов с ценой
n_with_price = prices_month_df["resort_id"].nunique()
print(f"Курортов с ценой: {n_with_price} из 32")

# 5. Сколько месяцев на курорт
print("\nМесяцев с ценой на курорт:")
print(prices_month_df.groupby("resort_id")["month"].nunique().describe())

# 6. Агрегация до одной строки на курорт
adult_skipass_avg_rub = (
    prices_month_df
    .assign(price_point=lambda d: (d["adult_price_min_rub"] + d["adult_price_max_rub"]) / 2)
    .groupby("resort_id")
    .agg(
        adult_skipass_avg_rub=("price_point", "mean"),
        price_months_count=("month", "nunique"),
    )
    .reset_index()
)
adult_skipass_avg_rub.head()

✅ Цены положительны, min <= max
,Дубликатов по (resort_id, month, tariff_product): 0
,Курортов с ценой: 9 из 32
,
,Месяцев с ценой на курорт:
,count    9.00000
,mean     4.00000
,std      2.12132
,min      1.00000
,25%      3.00000
,50%      4.00000
,75%      6.00000
,max      7.00000
,Name: month, dtype: float64


,resort_id,adult_skipass_avg_rub,price_months_count
0,R001,6028.750000,4
1,R002,4483.333333,6
2,R003,4100.000000,1
3,R005,3918.750000,4
4,R007,3465.000000,6


## 3.Климат

In [9]:
with open(RAW / "climate_monthly.json", encoding="utf-8-sig") as f:
    climate_raw = json.load(f)

print("type:", type(climate_raw))
print("элементов:", len(climate_raw))

climate_df = pd.DataFrame(climate_raw[1:], columns=climate_raw[0])

print("\nФорма:", climate_df.shape)
print("Колонки:", climate_df.columns.tolist())
climate_df.head()

type: <class 'list'>
,элементов: 193
,
,Форма: (192, 12)
,Колонки: ['resort_id', 'name_ru', 'region', 'month', 'month_name_ru', 'years_count', 'mean_temperature_c', 'avg_monthly_precipitation_mm', 'avg_monthly_snowfall_cm', 'avg_freezing_days', 'avg_snow_days', 'weather_model']


,resort_id,name_ru,region,month,month_name_ru,years_count,mean_temperature_c,avg_monthly_precipitation_mm,avg_monthly_snowfall_cm,avg_freezing_days,avg_snow_days,weather_model
0,R001,Роза Хутор,Краснодарский край,11,Ноябрь,5,7.113333,146.30,30.772,1.0,10.2,ERA5
1,R001,Роза Хутор,Краснодарский край,12,Декабрь,5,3.045806,153.62,81.844,5.8,16.4,ERA5
2,R001,Роза Хутор,Краснодарский край,1,Январь,5,0.806452,163.40,95.522,11.8,18.4,ERA5
3,R001,Роза Хутор,Краснодарский край,2,Февраль,5,1.063830,166.28,100.436,10.2,18.4,ERA5
4,R001,Роза Хутор,Краснодарский край,3,Март,5,4.254194,121.46,47.740,6.2,16.8,ERA5


## Аудит климата + агрегация до зимних признаков

In [10]:
# Числовые колонки
numeric_cols = ["mean_temperature_c", "avg_monthly_precipitation_mm",
                "avg_monthly_snowfall_cm", "avg_freezing_days", "avg_snow_days"]
for c in numeric_cols:
    climate_df[c] = pd.to_numeric(climate_df[c], errors="coerce")
climate_df["month"] = pd.to_numeric(climate_df["month"], errors="coerce")

# 1. Одна строка = курорт + месяц
dups = climate_df.duplicated(subset=["resort_id", "month"]).sum()
print(f"Дубликатов (resort_id, month): {dups}")

# 2. Морозные и снежные дни в разумных пределах
assert climate_df["avg_freezing_days"].dropna().between(0, 183).all()
assert climate_df["avg_snow_days"].dropna().between(0, 183).all()
print("✅ Дни в диапазоне 0-183")

# 3. Снегопад >= 0
assert (climate_df["avg_monthly_snowfall_cm"].dropna() >= 0).all()
print("✅ Снегопад >= 0")

# 4. Средние по зимним месяцам (ноябрь–апрель)
winter_months = [11, 12, 1, 2, 3, 4]
climate_winter = climate_df[climate_df["month"].isin(winter_months)]

climate_features = (
    climate_winter
    .groupby("resort_id")
    .agg(
        avg_snow_days_per_season=("avg_snow_days", "sum"),
        avg_freezing_days_per_season=("avg_freezing_days", "sum"),
        avg_snowfall_cm_per_season=("avg_monthly_snowfall_cm", "sum"),
    )
    .reset_index()
)
  
print(f"\nКурортов после агрегации: {climate_features['resort_id'].nunique()}")
climate_features.head()

Дубликатов (resort_id, month): 0
,✅ Дни в диапазоне 0-183
,✅ Снегопад >= 0
,
,Курортов после агрегации: 32


,resort_id,avg_snow_days_per_season,avg_freezing_days_per_season,avg_snowfall_cm_per_season
0,R001,92.6,35.2,390.082
1,R002,92.6,16.8,390.082
2,R003,92.6,18.6,390.082
3,R004,93.2,151.6,243.922
4,R005,107.6,81.6,523.194


## 4.Поездки по регионам 

In [11]:
trips_path = RAW / "rosstat_tourist_trips.csv"
print("Файл существует:", trips_path.exists())

trips_df = pd.read_csv(trips_path, sep=";", encoding="utf-8-sig", dtype=str)
print("\nКолонки:", trips_df.columns.tolist())
trips_df.head()

Файл существует: True
,
,Колонки: ['Ряд', '2022', '2023', '2024', '2025']


,Ряд,2022,2023,2024,2025
0,Алтайский край — январь,52432,124555,86785,111837
1,Алтайский край — январь-август,1182370,1574676,1223525,1316592
2,Алтайский край — январь-апрель,376047,597216,395478,470680
3,Алтайский край — январь-декабрь,NaN,1993763,1576852,1684877
4,Алтайский край — январь-декабрь*,1571744,NaN,NaN,NaN


In [12]:
# в ночёвках было "Ряд;2023;2024;2025" — первая колонка без имени
# приведём её к нормальному виду
first_col = trips_df.columns[0]
trips_df = trips_df.rename(columns={first_col: "row_label"})

# "Алтайский край — январь-декабрь" -> region + period
trips_df[["region_raw", "period"]] = trips_df["row_label"].str.split(" — ", n=1, expand=True)

print("Уникальных периодов:", trips_df["period"].unique())

# оставляем только годовые ряды
trips_year = trips_df[trips_df["period"] == "январь-декабрь"].copy()

# переводим в длинный формат
years = [c for c in trips_year.columns if c.isdigit()]  # ['2023','2024','2025']
trips_long = trips_year.melt(
    id_vars=["region_raw"],
    value_vars=years,
    var_name="year",
    value_name="trips"
)
trips_long["trips"] = pd.to_numeric(trips_long["trips"], errors="coerce")
trips_long["year"] = trips_long["year"].astype(int)
trips_long.head()

Уникальных периодов: ['январь' 'январь-август' 'январь-апрель' 'январь-декабрь'
, 'январь-декабрь*' 'январь-июль' 'январь-июнь' 'январь-май' 'январь-март'
, 'январь-ноябрь' 'январь-ноябрь*' 'январь-октябрь' 'январь-октябрь*'
, 'январь-сентябрь' 'январь-февраль']


,region_raw,year,trips
0,Алтайский край,2022,NaN
1,Амурская область,2022,NaN
2,Архангельская область (кроме Ненецкого автоном...,2022,NaN
3,Архангельская область,2022,NaN
4,Астраханская область,2022,NaN


In [13]:
# Первую колонку переименовываем
first_col = trips_df.columns[0]
trips_df = trips_df.rename(columns={first_col: "row_label"})

# Разбиваем на регион и период
trips_df[["region_raw", "period"]] = trips_df["row_label"].str.split(" — ", n=1, expand=True)

print("Уникальные периоды:")
print(sorted(trips_df["period"].dropna().unique()))


Уникальные периоды:
,['январь', 'январь-август', 'январь-апрель', 'январь-декабрь', 'январь-декабрь*', 'январь-июль', 'январь-июнь', 'январь-май', 'январь-март', 'январь-ноябрь', 'январь-ноябрь*', 'январь-октябрь', 'январь-октябрь*', 'январь-сентябрь', 'январь-февраль']


In [14]:
#Годовые ряды поездок → long

# Только годовые ряды (со звёздочкой тоже попадают — это годовые данные за неполный период)
trips_year = trips_df[trips_df["period"].str.startswith("январь-декабрь", na=False)].copy()

# Годы
years = [c for c in trips_year.columns if str(c).isdigit()]
print("Годы:", years)

# Длинный формат
trips_long = trips_year.melt(
    id_vars=["region_raw"],
    value_vars=years,
    var_name="year",
    value_name="trips",
)
trips_long["trips"] = pd.to_numeric(trips_long["trips"], errors="coerce")
trips_long["year"] = trips_long["year"].astype(int)

# Оставляем непустые
trips_long = trips_long.dropna(subset=["trips"]).reset_index(drop=True)

# Проверки
print("\nПропуски:", trips_long["trips"].isna().sum())
print("Trips > 0:", (trips_long["trips"] > 0).all())
print("Дубликаты (region, year):", trips_long.duplicated(subset=["region_raw", "year"]).sum())

trips_long.head()

Годы: ['2022', '2023', '2024', '2025']
,
,Пропуски: 0
,Trips > 0: True
,Дубликаты (region, year): 0


,region_raw,year,trips
0,Алтайский край,2022,1571744.0
1,Амурская область,2022,446955.0
2,Архангельская область (кроме Ненецкого автоном...,2022,517315.0
3,Архангельская область,2022,534943.0
4,Астраханская область,2022,414919.0


## 5.Ночёвки по регионам

In [15]:
nights_df = pd.read_csv(
    RAW / "rosstat_tourist_nights.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype=str,
)
print(nights_df.columns.tolist())
nights_df.head()

['Ряд', '2023', '2024', '2025']


,Ряд,2023,2024,2025
0,Алтайский край — январь-декабрь,6033880,6524855,7744587
1,Алтайский край — январь-июнь,2545385,2868046,2689398
2,Алтайский край — январь-март,1010314,1259104,1059474
3,Алтайский край — январь-сентябрь,5222692,5686807,5745536
4,Амурская область — январь-декабрь,1496124,2085918,1956217


In [16]:
first_col = nights_df.columns[0]
nights_df = nights_df.rename(columns={first_col: "row_label"})
nights_df[["region_raw", "period"]] = nights_df["row_label"].str.split(" — ", n=1, expand=True)

# Только годовые ряды
nights_year = nights_df[nights_df["period"].str.startswith("январь-декабрь", na=False)].copy()

years = [c for c in nights_year.columns if str(c).isdigit()]
nights_long = nights_year.melt(
    id_vars=["region_raw"],
    value_vars=years,
    var_name="year",
    value_name="nights",
)
nights_long["nights"] = pd.to_numeric(nights_long["nights"], errors="coerce")
nights_long["year"] = nights_long["year"].astype(int)
nights_long = nights_long.dropna(subset=["nights"]).reset_index(drop=True)

print("Пропуски:", nights_long["nights"].isna().sum())
print("Nights > 0:", (nights_long["nights"] > 0).all())
nights_long.head()

Пропуски: 0
,Nights > 0: True


,region_raw,year,nights
0,Алтайский край,2023,6033880
1,Амурская область,2023,1496124
2,Архангельская область (кроме Ненецкого автоном...,2023,2452972
3,Архангельская область,2023,2568737
4,Астраханская область,2023,2575882


 # 6.Словарь регионов (region_key) и соединение с курортами

In [17]:
# Целевой список регионов — из справочника курортов
regions_target = sorted(resorts_df["region"].unique())
print(f"Регионов в справочнике: {len(regions_target)}")
for r in regions_target:
    print(" ", r)

# Список регионов из ЕМИСС
regions_source = sorted(set(trips_long["region_raw"].unique()) | set(nights_long["region_raw"].unique()))
print(f"\nРегионов в ЕМИСС: {len(regions_source)}")

# Чего не хватает
missing = sorted(set(regions_target) - set(regions_source))
print("\nНе нашлись в ЕМИСС:")
for m in missing:
    print(" ", m)

Регионов в справочнике: 20
,  Иркутская область
,  Кабардино-Балкарская Республика
,  Камчатский край
,  Карачаево-Черкесская Республика
,  Кемеровская область
,  Краснодарский край
,  Красноярский край
,  Ленинградская область
,  Московская область
,  Мурманская область
,  Пермский край
,  Республика Алтай
,  Республика Башкортостан
,  Республика Северная Осетия — Алания
,  Республика Хакасия
,  Самарская область
,  Сахалинская область
,  Свердловская область
,  Хабаровский край
,  Челябинская область
,
,Регионов в ЕМИСС: 96
,
,Не нашлись в ЕМИСС:
,  Кемеровская область
,  Республика Северная Осетия — Алания


In [18]:
# ищем похожие названия в ЕМИСС
for pattern in ["Кемеров", "Осети", "Алани"]:
    matches = [r for r in regions_source if pattern.lower() in r.lower()]
    print(f"\nПохожие на '{pattern}':")
    for m in matches:
        print(" ", repr(m))


,Похожие на 'Кемеров':
,  'Кемеровская область - Кузбасс'
,
,Похожие на 'Осети':
,  'Республика Северная Осетия - Алания'
,
,Похожие на 'Алани':
,  'Республика Северная Осетия - Алания'


In [19]:
# Заполняем вручную по результатам предыдущей ячейки.
# Слева — как регион называется в ЕМИСС, справа — как в справочнике курортов.
region_map = {
    "Кемеровская область - Кузбасс": "Кемеровская область",
    "Республика Северная Осетия - Алания": "Республика Северная Осетия — Алания",
}

def to_region_key(name):
    if pd.isna(name):
        return name
    return region_map.get(name, name)

In [20]:
# Региональные метрики (широкий формат)

# Широкий формат: по годам
trips_wide = trips_long.pivot(index="region_raw", columns="year", values="trips").reset_index()
nights_wide = nights_long.pivot(index="region_raw", columns="year", values="nights").reset_index()

# Переименование колонок
trips_wide = trips_wide.rename(columns={
    2022: "trips_2022", 2023: "trips_2023", 2024: "trips_2024", 2025: "trips_2025",
})
nights_wide = nights_wide.rename(columns={
    2023: "nights_2023", 2024: "nights_2024", 2025: "nights_2025",
})

# Соединение
region_metrics = trips_wide.merge(nights_wide, on="region_raw", how="outer")

# region_key
region_metrics["region_key"] = region_metrics["region_raw"].map(to_region_key)

# Метрики
region_metrics["trips_growth_2025"] = region_metrics["trips_2025"] / region_metrics["trips_2024"] - 1
region_metrics["avg_stay_nights_2025"] = region_metrics["nights_2025"] / region_metrics["trips_2025"]

print("Форма region_metrics:", region_metrics.shape)
print("Колонки:", region_metrics.columns.tolist())
region_metrics.head()

Форма region_metrics: (96, 11)
,Колонки: ['region_raw', 'trips_2022', 'trips_2023', 'trips_2024', 'trips_2025', 'nights_2023', 'nights_2024', 'nights_2025', 'region_key', 'trips_growth_2025', 'avg_stay_nights_2025']


year,region_raw,trips_2022,trips_2023,trips_2024,trips_2025,nights_2023,nights_2024,nights_2025,region_key,trips_growth_2025,avg_stay_nights_2025
0,Алтайский край,1571744.0,1993763.0,1576852.0,1684877.0,6033880,6524855,7744587,Алтайский край,0.068507,4.596530
1,Амурская область,446955.0,422839.0,453946.0,667679.0,1496124,2085918,1956217,Амурская область,0.470834,2.929876
2,Архангельская область,534943.0,830290.0,772772.0,638512.0,2568737,2173242,2634827,Архангельская область,-0.173738,4.126511
3,Архангельская область (кроме Ненецкого автоном...,517315.0,805466.0,741696.0,605594.0,2452972,2095315,2546836,Архангельская область (кроме Ненецкого автоном...,-0.183501,4.205517
4,Астраханская область,414919.0,508941.0,699335.0,748188.0,2575882,2732121,2126012,Астраханская область,0.069856,2.841548


In [21]:
# Соединение региональных метрик с курортами (many-to-one)

resorts_with_region = resorts_df.merge(
    region_metrics[["region_key", "trips_2025", "trips_growth_2025", "avg_stay_nights_2025"]],
    left_on="region", right_on="region_key", how="left"
)

# Проверки ТЗ
assert len(resorts_with_region) == 32, f"Потеряли курорты: {len(resorts_with_region)}"
assert resorts_with_region["resort_id"].nunique() == 32, "resort_id не уникален"

n_missing = resorts_with_region["trips_2025"].isna().sum()
print(f"Курортов без trips_2025: {n_missing}")

if n_missing > 0:
    print("\n⚠️ Курорты без региональных данных:")
    print(resorts_with_region.loc[
        resorts_with_region["trips_2025"].isna(),
        ["resort_id", "name_ru", "region"]
    ])

print("\nПример соединения:")
print(resorts_with_region[["resort_id", "name_ru", "region", "trips_2025",
                            "trips_growth_2025", "avg_stay_nights_2025"]].head(10))

Курортов без trips_2025: 0
,
,Пример соединения:
,  resort_id         name_ru                               region  trips_2025  trips_growth_2025  avg_stay_nights_2025
,0      R001      Роза Хутор                   Краснодарский край  18427326.0          -0.038392              6.005796
,1      R002  Красная Поляна                   Краснодарский край  18427326.0          -0.038392              6.005796
,2      R003  Газпром Поляна                   Краснодарский край  18427326.0          -0.038392              6.005796
,3      R004         Шерегеш                  Кемеровская область   1266539.0          -0.126286              4.808109
,4      R005           Архыз      Карачаево-Черкесская Республика   1303991.0           0.863371              1.748027
,5      R006          Домбай      Карачаево-Черкесская Республика   1303991.0           0.863371              1.748027
,6      R007         Эльбрус      Кабардино-Балкарская Республика    341320.0          -0.168919              7.831879

## Готовая витрина и контрольные баллы

In [22]:
with open(PROCESSED / "model_data.json", encoding="utf-8-sig") as f:
    model_data = json.load(f)

print("Ключи верхнего уровня:", list(model_data.keys()))
print("Веса:", model_data["weights"])
print("Курортов в model_rows:", len(model_data["model_rows"]))

model_df = pd.DataFrame(model_data["model_rows"])
print("Форма model_df:", model_df.shape)
model_df.head(10)

Ключи верхнего уровня: ['weights', 'regions', 'resorts', 'model_rows']
,Веса: {'region_trips': 25.0, 'region_growth': 10.0, 'avg_stay': 5.0, 'pistes': 10.0, 'lifts': 7.0, 'lodging': 5.0, 'amenities': 3.0, 'snow_days': 8.0, 'freezing_days': 6.0, 'snowfall': 6.0, 'airport': 8.0, 'transport': 2.0, 'price': 5.0}
,Курортов в model_rows: 32
,Форма model_df: (32, 43)


,resort_id,name_ru,region,piste_segments_6km,lift_lines_6km,lodging_objects_6km,food_objects_6km,attraction_objects_6km,transport_objects_6km,avg_snowfall_cm_per_season,avg_freezing_days_per_season,avg_snow_days_per_season,nearest_airport_distance_km,adult_skipass_avg_rub,price_months_count,completeness_pct,coordinate_quality,quality_flags,trips_2022,trips_2023,trips_2024,trips_2025,nights_2023,nights_2024,nights_2025,trips_growth_2025,avg_stay_nights_2025,region_trips_score,region_growth_score,avg_stay_score,pistes_score,lifts_score,lodging_score,amenities_score,snow_days_score,freezing_days_score,snowfall_score,airport_score,transport_score,price_score,popularity_score_default,confidence_score,rank_default
0,R003,Газпром Поляна,Краснодарский край,240,63,223,243,77,77,390.082,18.6,92.6,36.9,4100.000000,1,1.000000,resort_or_complex_page,OSM radii overlap,20121753.0,19305168.0,19163040.0,18427326.0,106687870.0,115150411.0,110670767.0,-0.038392,6.005796,95.844308,42.321531,83.429939,94.335635,99.628588,97.591937,98.913939,69.117647,1.152369,81.739358,40.300405,71.470378,40.262497,73.788787,100.000000,1
1,R002,Красная Поляна,Краснодарский край,121,44,255,259,82,76,390.082,16.8,92.6,35.9,4483.333333,6,1.000000,resort_or_complex_page,OSM radii overlap,20121753.0,19305168.0,19163040.0,18427326.0,106687870.0,115150411.0,110670767.0,-0.038392,6.005796,95.844308,42.321531,83.429939,82.626646,91.190928,100.000000,100.000000,69.117647,0.000000,81.739358,40.896180,71.258701,30.929028,71.687850,100.000000,2
2,R001,Роза Хутор,Краснодарский край,334,64,86,107,42,45,390.082,35.2,92.6,37.3,6028.750000,4,1.000000,nearby_feature_proxy,coordinates are proxy; OSM radii overlap,20121753.0,19305168.0,19163040.0,18427326.0,106687870.0,115150411.0,110670767.0,-0.038392,6.005796,95.844308,42.321531,83.429939,100.000000,100.000000,80.536794,85.874856,69.117647,11.779770,81.739358,40.066486,62.807605,0.000000,71.569843,75.000000,3
3,R030,Сорочаны,Московская область,43,17,30,34,6,114,137.550,105.6,78.8,34.6,NaN,0,1.000000,resort_or_complex_page,None,15211970.0,16456683.0,18009855.0,22431311.0,73669658.0,100751338.0,96458300.0,0.245502,4.300163,100.000000,99.100406,55.002721,65.086080,69.240623,61.927454,63.645116,43.750000,56.850192,30.312379,41.695295,77.839053,50.000000,70.143513,75.000000,4
4,R005,Архыз,Карачаево-Черкесская Республика,42,13,187,69,25,18,523.194,81.6,107.6,99.4,3918.750000,4,1.000000,resort_or_complex_page,None,743617.0,404188.0,699802.0,1303991.0,2856842.0,2155461.0,2279411.0,0.863371,1.748027,39.869439,100.000000,12.467111,64.690672,63.220232,94.432361,78.046694,96.691176,41.485275,96.257483,18.594372,48.302554,44.983908,59.250873,100.000000,5
5,R006,Домбай,Карачаево-Черкесская Республика,30,13,85,51,12,5,564.312,86.0,106.4,136.0,NaN,0,1.000000,resort_or_complex_page,None,743617.0,404188.0,699802.0,1303991.0,2856842.0,2155461.0,2279411.0,0.863371,1.748027,39.869439,100.000000,12.467111,59.062781,63.220232,80.328309,71.277086,94.485294,44.302177,100.000000,11.669136,29.393226,50.000000,57.315487,75.000000,6
6,R029,Игора,Ленинградская область,21,9,39,28,12,98,127.498,112.0,75.8,79.5,NaN,0,1.000000,nearby_feature_proxy,coordinates are proxy,15205197.0,10505620.0,8966335.0,8884916.0,32350165.0,32357511.0,34413126.0,-0.009081,3.873208,80.426525,48.183896,47.886797,53.164311,55.159834,66.524101,63.645116,38.235294,60.947503,26.579676,23.516274,75.381432,50.000000,55.931923,50.000000,7
7,R004,Шерегеш,Кемеровская область,142,23,233,57,4,54,243.922,151.6,93.2,119.2,NaN,0,1.000000,resort_or_complex_page,None,1131655.0,1174177.0,1449604.0,1266539.0,5185409.0,4731671.0,6089657.0,-0.126286,4.808109,39.253523,24.742757,63.468476,85.358328,76.132223,98.379559,70.732961,70.220588,86.299616,58.546728,14.583975,65.738985,50.000000,55.657061,75.000000,8
8,R016,Горный воздух,Сахалинская область,54,7,51,146,75,268,204.330,150.8,81.8,9.5,NaN,0,0.909091,manual_review_required,check coordinates,302739.0,514905.0,289585.0,348282.0,921582.0,1028972.0,1224528.0

In [23]:
# 32 строки и 32 уникальных resort_id
assert len(model_df) == 32, f"Строк: {len(model_df)}"
assert model_df["resort_id"].nunique() == 32, f"Уникальных ID: {model_df['resort_id'].nunique()}"
print("✅ 32 строки, 32 уникальных resort_id")

# Все непустые баллы в [0, 100]
score_cols = [c for c in model_df.columns if c.endswith("_score")]
print(f"Проверяю баллы: {len(score_cols)} колонок")

problems = []
for c in score_cols:
    s = model_df[c].dropna() # <-- убираем NaN перед проверкой
    if not s.between(-0.01, 100.01).all():
        bad = model_df.loc[~model_df[c].between(-0.01, 100.01), ["resort_id", "name_ru", c]]
        problems.append((c, bad))

if problems:
    print("\n⚠️ Колонки с проблемами:")
    for c, bad in problems:
        print(f"\n--- {c} ---")
        print(bad.to_string(index=False))
else:
    print("✅ Все непустые баллы в [0, 100]")

# Сумма весов = 100
w = model_data["weights"]
total = sum(w.values())
assert abs(total - 100) < 0.001, f"Сумма весов: {total}"
print(f"✅ Сумма весов: {total}")

# Топ-10
print("\nТоп-10 (сверить с приложением Б ТЗ):")
print(
    model_df
    .sort_values("rank_default")
    [["rank_default", "name_ru", "region", "popularity_score_default", "confidence_score"]]
    .head(10)
    .to_string(index=False)
)

✅ 32 строки, 32 уникальных resort_id
,Проверяю баллы: 14 колонок
,✅ Все непустые баллы в [0, 100]
,✅ Сумма весов: 100.0
,
,Топ-10 (сверить с приложением Б ТЗ):
, rank_default        name_ru                          region  popularity_score_default  confidence_score
,            1 Газпром Поляна              Краснодарский край                 73.788787        100.000000
,            2 Красная Поляна              Краснодарский край                 71.687850        100.000000
,            3     Роза Хутор              Краснодарский край                 71.569843         75.000000
,            4       Сорочаны              Московская область                 70.143513         75.000000
,            5          Архыз Карачаево-Черкесская Республика                 59.250873        100.000000
,            6         Домбай Карачаево-Черкесская Республика                 57.315487         75.000000
,            7          Игора           Ленинградская область                 55.931923         50